# ARCHIVE — bank-statement run (F1 0.026). Do not run.

Kept as evidence for the comparison in [`../FINDINGS.md`](../FINDINGS.md). This notebook
clones `github.com/aashishkumar-tech/document-extraction-finetune` and imports its `src/`,
so it **will not execute from this repo**. Outputs are preserved as-is.

Superseded by [`donut_cord_receipts.ipynb`](donut_cord_receipts.ipynb) — same model, same
LoRA recipe, legible inputs, **F1 0.901**.

Caveat on the 0.026 number: the checkpoint behind it was produced by the hand-rolled LoRA
fold that was later proven to silently discard trained weights (see FINDINGS.md §"the save
bug"). The true score of that training run is unknown and was probably higher.


# Donut LoRA Fine-Tuning — Replica + Test Notebook

Replica of the original Kaggle notebook (`code.md`), reorganized into clean runnable cells
(no dash separators), with added **diagnostic** and **fix** cells so you can test the
0.000-F1 inference bug end to end.

**Environment:** Kaggle 2× Tesla T4 (`CUDA_VISIBLE_DEVICES=0` pins to one GPU).

**Flow:**
1. Imports / env
2. Replicated original cells (setup → data → model → LoRA → train → save)
3. In-session generation **smoke test** (proves whether trained weights are good)
4. **Fixed** `predict_donut.py` (re-applies pad / decoder_start / eos config)
5. Run inference + scorer
6. Assertions to validate the fix

## 1. Import Required Libraries & Pin GPU

Pin to a single T4 and confirm the GPU is visible before doing anything heavy.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Confirm GPU visibility (Kaggle: expect Tesla T4)
!nvidia-smi


## 2. Replicate Original Code

Exact replica of the original Kaggle cells (setup, data prep, model, LoRA, training, save).
These are the baseline — unchanged behavior.

### 2.1 Clone repo & system deps

In [ ]:
%cd /kaggle/working
!git clone https://github.com/aashishkumar-tech/document-extraction-finetune.git
%cd document-extraction-finetune
!apt-get install -y poppler-utils -q


In [ ]:
# 2.2 Python deps
!pip install -q huggingface_hub transformers peft accelerate bitsandbytes datasets \
    opencv-python-headless pdf2image pillow python-Levenshtein pandas python-dotenv


### 2.2b Persistent data/model backup (survives session timeouts)

Run the **config** cell once (set your HF token + repo names). Then:
- The **restore** cell pulls prepared data from HuggingFace in seconds — if it succeeds,
  you can SKIP the slow download/degrade cells (2.3, 2.4).
- After prep, the **backup-data** cell uploads the prepared data once.
- After training, the **backup-model** cell uploads the model so a timeout never loses it.

In [ ]:
# 2.2b-config: set your HuggingFace token + repo names (edit these once)
import os

HF_TOKEN     = os.environ.get("HF_TOKEN", "hf_xxx_replace_me")  # or set via Kaggle/HF secret
HF_USERNAME  = "your-username"                                   # <-- change
DATA_REPO    = f"{HF_USERNAME}/donut-bank-data"                  # dataset repo (prepared data)
MODEL_REPO   = f"{HF_USERNAME}/donut-bank-model"                 # model repo (fine-tuned)
# VERSIONED archive name: the restore/guard cells fetch THIS filename. The old build was
# uploaded as donut_data.tar.gz, so reusing that name would keep pulling the STALE data.
# Bumping the version forces a fresh build+upload of the new --all (~800-doc) dataset.
DATA_ARCHIVE = "donut_data_v2_all.tar.gz"
DATA_DIRS    = "data/degraded data/splits data/donut_ready"     # what gets backed up

from huggingface_hub import login
try:
    login(HF_TOKEN)
    print("HF login OK. DATA_REPO:", DATA_REPO, "| MODEL_REPO:", MODEL_REPO)
except Exception as e:
    print("HF login failed (set HF_TOKEN):", e)


In [ ]:
# 2.2b-restore: try to pull prepared data from HF. If it works, SKIP cells 2.3 and 2.4.
from huggingface_hub import hf_hub_download

DATA_READY = False
try:
    archive_path = hf_hub_download(
        repo_id=DATA_REPO, filename=DATA_ARCHIVE, repo_type="dataset",
    )
    !tar -xzf {archive_path}
    DATA_READY = True
    print("Restored prepared data from HF in seconds. -> SKIP cells 2.3 and 2.4, go to model load.")
except Exception as e:
    print("No backup found yet (first run). Run 2.3 + 2.4 to build data, then run 2.2b-backup.")
    print("Reason:", e)


In [ ]:
# 2.3 Download source docs & generate degraded variants
# NOTE: prefer the first-time-only cell 2.3b (it guards on the HF backup). This cell is the
# manual/standalone equivalent. Use --all: the AgamiAI repo has only ~200 PDFs per folder
# x 4 folders = ~800 total, so '--n' above that just builds non-matching patterns (no more
# files). --all grabs the whole dataset. Check disk first: !df -h /kaggle/working
!python -m src.data.download_all --all
!python -m src.data.batch_degrade


In [ ]:
# 2.4 Build splits + Donut-ready jsonl manifests
!python -m src.data.prepare_dataset
!python -m src.data.prepare_donut_dataset


In [ ]:
# 2.2b-backup: upload prepared data to HF ONCE (run only after 2.3 + 2.4 built the data).
# Next session, the restore cell pulls this in seconds instead of the ~2h rebuild.
from huggingface_hub import HfApi

if not DATA_READY:  # only upload if we built fresh data this session
    !tar -czf {DATA_ARCHIVE} {DATA_DIRS}
    api = HfApi()
    api.create_repo(DATA_REPO, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=DATA_ARCHIVE,
        path_in_repo=DATA_ARCHIVE,
        repo_id=DATA_REPO,
        repo_type="dataset",
    )
    print(f"Uploaded prepared data to https://huggingface.co/datasets/{DATA_REPO} — now permanent.")
else:
    print("Data was restored from backup; no need to re-upload.")


In [ ]:
# 2.3b FIRST-TIME-ONLY: build the 400-doc / 1,600-image dataset and back it up to HF.
# Guard: if the archive already exists on HF (DATA_REPO), skip the whole ~2h build+upload.
# So this cell is safe to leave in place — it runs the heavy work only on the FIRST session.
from huggingface_hub import HfApi, hf_hub_download

def _hf_data_exists():
    try:
        hf_hub_download(repo_id=DATA_REPO, filename=DATA_ARCHIVE, repo_type="dataset")
        return True
    except Exception:
        return False

if _hf_data_exists():
    print(f"Dataset already on HF ({DATA_REPO}/{DATA_ARCHIVE}) -> skipping build+upload.")
    print("Use the 2.2b-restore cell to pull it in seconds.")
else:
    print("No HF backup found -> FIRST-TIME build, then upload.\n")
    print("NOTE: the AgamiAI repo has 100 PDFs per folder x 4 folders = 400 total docs.")
    print("      batch_degrade makes 4 tiers/doc -> 1,600 degraded images.")
    print("      '--n' is a request ceiling, not a guarantee: values above the real count")
    print("      just generate non-matching patterns. So we use --all to grab everything.\n")
    !df -h /kaggle/working
    # 1) generate source docs (--all = whole dataset, 400 pairs) + degraded variants
    !python -m src.data.download_all --all
    !python -m src.data.batch_degrade
    # how many real PDFs per folder did we actually get?
    !for d in data/raw/train/*/; do echo "$d: $(ls "$d"/*.pdf 2>/dev/null | wc -l) pdfs"; done
    # 2) build splits + Donut-ready manifests
    !python -m src.data.prepare_dataset
    !python -m src.data.prepare_donut_dataset
    # 3) archive + upload to HF (permanent)
    !tar -czf {DATA_ARCHIVE} {DATA_DIRS}
    api = HfApi()
    api.create_repo(DATA_REPO, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=DATA_ARCHIVE,
        path_in_repo=DATA_ARCHIVE,
        repo_id=DATA_REPO,
        repo_type="dataset",
    )
    print(f"\nUploaded dataset (400 docs / 1,600 images, whole AgamiAI repo) to https://huggingface.co/datasets/{DATA_REPO} — now permanent.")


In [ ]:
# 2.5 Load donut-base, add 22 custom task tags, resize embeddings
import torch
from transformers import DonutProcessor, VisionEncoderDecoderModel

MODEL_NAME = "naver-clova-ix/donut-base"
processor = DonutProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

HEADER_FIELDS = [
    "account_number", "ifsc_code", "micr_code", "bank_name", "account_holder",
    "opening_balance", "closing_balance", "start_date", "end_date", "statement_date",
]
new_tokens = ["<s_extraction>", "</s_extraction>"]
for field in HEADER_FIELDS:
    new_tokens += [f"<s_{field}>", f"</s_{field}>"]
processor.tokenizer.add_tokens(new_tokens)
model.decoder.resize_token_embeddings(len(processor.tokenizer))
processor.image_processor.size = {"height": 1280, "width": 960}

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Model loaded on:", device)


In [ ]:
# 2.6 Set pad / decoder_start token ids on the training model config
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
if model.config.decoder_start_token_id is None:
    model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s>")
print("pad_token_id:", model.config.pad_token_id)
print("decoder_start_token_id:", model.config.decoder_start_token_id)


In [ ]:
# 2.7 Confirm module names used by LoRA modules_to_save
for name, module in model.named_modules():
    if "embed_tokens" in name or "lm_head" in name:
        print(name)


## 3. Prepare Sample / Test Data

Build the `Dataset` objects from the Donut-ready jsonl manifests. `train_dataset[0]` later
doubles as the **sample input** for the in-session smoke test.

In [ ]:
# 3.1 Dataset that yields pixel_values + label token ids
import json
from PIL import Image
from torch.utils.data import Dataset

MAX_LENGTH = 512

class DonutDataset(Dataset):
    def __init__(self, jsonl_path, processor, max_length=MAX_LENGTH):
        self.processor = processor
        self.max_length = max_length
        with open(jsonl_path, "r") as f:
            self.records = [json.loads(line) for line in f]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        image = Image.open(record["image_path"]).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(
            record["target_sequence"], add_special_tokens=False,
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt",
        ).input_ids.squeeze()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

train_dataset = DonutDataset("data/donut_ready/train.jsonl", processor)
val_dataset = DonutDataset("data/donut_ready/val.jsonl", processor)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")


In [ ]:
# 3.1b FOCUSED RETRAIN DATA — rebuild train/val manifests on the SIGNAL, not the noise.
# The 580-doc eval proved two things:
#   (1) accuracy only exists on the CLEAN tier (auto_F1 0.054 there vs ~0 on light/med/heavy),
#   (2) free-text fields (account_holder, bank_name) produce huge false-positive counts and
#       drag precision to the floor, while ID/date/balance fields carry all the real signal.
# Training on medium/heavy images and on the FP-magnet fields just injects noise into the
# loss. This cell rebuilds FILTERED manifests so the model spends its (small) capacity only
# on the tiers it can serve and the fields that actually score. Existing 3.3 -> 3.6 cells
# then retrain UNCHANGED on these filtered files.
import json, re
from pathlib import Path

# ---- knobs -----------------------------------------------------------------
TRAIN_LEVELS = {"clean", "light"}          # drop medium/heavy: we DEFER them at serve time
KEEP_FIELDS = [                            # high-signal only (balances kept; edit to taste)
    "account_number", "ifsc_code", "micr_code",
    "start_date", "end_date", "statement_date",
    "opening_balance", "closing_balance",
]
# For an ID/date-only first pass, set:
# KEEP_FIELDS = ["account_number","ifsc_code","micr_code","start_date","end_date","statement_date"]
DROP_FIELDS = [f for f in HEADER_FIELDS if f not in KEEP_FIELDS]
print("KEEP:", KEEP_FIELDS)
print("DROP:", DROP_FIELDS, "\n")

SRC_DIR = Path("data/donut_ready")
DST_DIR = Path("data/donut_ready_focused")
DST_DIR.mkdir(parents=True, exist_ok=True)

def _level_of(rec):
    """Get degradation tier from an explicit key, else infer from the image path."""
    lvl = rec.get("level")
    if lvl:
        return str(lvl).lower()
    p = str(rec.get("image_path", "")).lower()
    for t in ("clean", "light", "medium", "heavy"):
        if t in p:
            return t
    return "unknown"

def _filter_sequence(seq):
    """Keep only the <s_field>value</s_field> blocks for fields in KEEP_FIELDS,
    preserving KEEP_FIELDS order, re-wrapped in the extraction tags. Returns None
    if no kept field is present (record becomes useless -> skip it)."""
    kept = []
    for f in KEEP_FIELDS:
        m = re.search(rf"<s_{re.escape(f)}>(.*?)</s_{re.escape(f)}>", seq, flags=re.DOTALL)
        if m and m.group(1).strip():
            kept.append(f"<s_{f}>{m.group(1)}</s_{f}>")
    if not kept:
        return None
    return f"<s_extraction>{''.join(kept)}</s_extraction>"

def _rebuild(split):
    src = SRC_DIR / f"{split}.jsonl"
    dst = DST_DIR / f"{split}.jsonl"
    kept_n = dropped_level = dropped_empty = 0
    with open(src) as fin, open(dst, "w") as fout:
        for line in fin:
            rec = json.loads(line)
            if _level_of(rec) not in TRAIN_LEVELS:
                dropped_level += 1
                continue
            new_seq = _filter_sequence(rec["target_sequence"])
            if new_seq is None:
                dropped_empty += 1
                continue
            rec["target_sequence"] = new_seq
            fout.write(json.dumps(rec) + "\n")
            kept_n += 1
    print(f"[{split}] kept={kept_n}  dropped(level)={dropped_level}  dropped(no-kept-field)={dropped_empty}"
          f"  -> {dst}")
    return kept_n

n_tr = _rebuild("train")
n_va = _rebuild("val")

# Re-point the datasets at the focused manifests so 3.3 -> 3.6 retrain on them unchanged.
train_dataset = DonutDataset(str(DST_DIR / "train.jsonl"), processor)
val_dataset   = DonutDataset(str(DST_DIR / "val.jsonl"), processor)
print(f"\nFocused Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print("NOTE: tokenizer/vocab is unchanged (still 57547) — we only removed field BLOCKS from")
print("the targets, so the model simply never emits the dropped tags. No 2.5 change needed.")
print("NEXT: run 3.3 -> 3.4 -> 3.5 (train) -> 3.6 (save) -> 3.6b (upload) -> 4.2 -> 5.1.")

In [ ]:
# 3.2 Checkpoint dir
import os

CHECKPOINT_DIR = "/kaggle/working/checkpoints/donut"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")


In [ ]:
# 3.3 Wrap model with LoRA (embed_tokens + lm_head saved fully)
!pip install -q -U torchao

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query", "value", "q_proj", "v_proj"],
    modules_to_save=["embed_tokens", "lm_head"],
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# 3.4 Enable input-embedding grads for gradient checkpointing
from transformers.modeling_utils import PreTrainedModel
from peft import PeftModel

def _noop(self): pass
PreTrainedModel.enable_input_require_grads = _noop
PeftModel.enable_input_require_grads = _noop

def make_inputs_require_grads(module, input, output):
    if isinstance(output, tuple):
        output[0].requires_grad_(True)
    else:
        output.requires_grad_(True)

model._require_grads_hook = model.get_input_embeddings().register_forward_hook(make_inputs_require_grads)
model.gradient_checkpointing_enable()
print("Hook registered")


In [ ]:
# 3.5 Train (Seq2SeqTrainer). ~1.5h on T4. Keeps the LAST epoch (val_loss != F1 for extraction).
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])
    return {"pixel_values": pixel_values, "labels": labels}

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/checkpoints/donut_v2",
    num_train_epochs=12,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=1.5e-4,               # lower than 3e-4: less overfitting, best epoch comes later
    warmup_steps=50,                    # warmup_ratio is deprecated -> use warmup_steps
    weight_decay=0.05,                  # stronger reg: delays overfitting so best epoch has learned more
    fp16=False,                         # CRITICAL FIX: fp16 caused loss=47 / 0.000 F1 on T4
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=False,       # KEEP LAST epoch, not lowest-val-loss: val_loss != F1 for extraction
    logging_steps=25,
    predict_with_generate=False,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    data_collator=collate_fn,
)

train_result = trainer.train()


In [ ]:
# 3.5-DEBUG — FAST overfit test (30 images, 2 epochs, ~5 min). Use this to iterate cheaply.
# Goal: if the model can memorize 30 images, the training setup is sound.
# If PRED stays garbage even here, the bug is in the training path (grad flow), not the data.
from torch.utils.data import Subset
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

N_DEBUG = min(30, len(train_dataset))
debug_train = Subset(train_dataset, list(range(N_DEBUG)))
debug_val   = Subset(val_dataset, list(range(min(5, len(val_dataset)))))

# Cell 3.4 turned gradient checkpointing ON; disable it for the fp32 debug run.
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()
model.config.use_cache = True

def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.stack([item["labels"] for item in batch])
    return {"pixel_values": pixel_values, "labels": labels}

debug_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/checkpoints/donut_debug",
    num_train_epochs=30,                # enough steps to actually overfit 30 images
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,      # more optimizer steps per epoch
    gradient_checkpointing=False,       # 30 imgs fit without it -> removes the grad-hook variable
    learning_rate=5e-4,                 # higher LR so the tags are learned fast
    fp16=False,                         # T4 has no bf16; fp32 avoids the loss=47 fp16 blow-up
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=5,
    predict_with_generate=False,
    report_to="none",
    remove_unused_columns=False,
)

debug_trainer = Seq2SeqTrainer(
    model=model, args=debug_args,
    train_dataset=debug_train, eval_dataset=debug_val,
    data_collator=collate_fn,
)
debug_trainer.train()
print("\nDebug train done. Now run CHECK C (cell 4.1b) to see PRED vs GOLD on this tiny set.")


In [ ]:
# 3.5b IS THE KERNEL STILL ALIVE? Run this BEFORE deciding to retrain.
# If all three checks print OK, the trained model is still in memory and the GPU is
# healthy -> just run the fixed save cell (3.6) then inference. NO retrain needed.
# The earlier CUDA asserts happened in the `!python ...` SUBPROCESS, not this kernel,
# so this kernel's GPU context is usually fine.
import torch

# (1) trained PEFT model object still exists?
assert "model" in globals(), "FAIL: 'model' is gone -> kernel was restarted. You must retrain."
print("OK (1): trained `model` object is in memory.")

# (2) is it still PEFT-wrapped, or was it already merged by an earlier 3.6 run?
MODEL_IS_MERGED = not any("modules_to_save.default.weight" in k for k in model.state_dict())
if MODEL_IS_MERGED:
    print("NOTE (2): model was ALREADY merged by an earlier save attempt (no PEFT wrappers left).")
    print("          -> The save cell 3.6 handles this. But FIRST run CHECK C (cell 4.1b) to")
    print("             confirm the in-memory weights still read images (PRED ~= GOLD).")
else:
    print("OK (2): PEFT modules_to_save (trained tag embeddings) present.")

# (3) GPU context not poisoned? A tiny op raises if the device-side assert killed it.
try:
    _ = (torch.ones(8, device="cuda") * 2).sum().item()
    print("OK (3): GPU context is healthy.")
    if MODEL_IS_MERGED:
        print("    NEXT: run CHECK C (4.1b). If PRED matches GOLD -> run 3.6 (saves as-is).")
        print("          If PRED is garbage -> the earlier merge corrupted it -> retrain.")
    else:
        print("    NEXT: run cell 3.6 (save), no retrain.")
except RuntimeError as e:
    print("FAIL (3): GPU context poisoned:", e)
    print("-> Restart the kernel and retrain (cells 3.3 -> 3.4 -> 3.5 -> 3.6).")


In [ ]:
# 3.6 Save model — RECONSTRUCT a clean plain VisionEncoderDecoderModel.
#   The in-memory `model` is a PEFT/merged hybrid whose param names differ from a
#   plain HF model (extra prefixes like base_model.model. / .base_layer, or merged
#   in place). Rather than guess the exact renaming, we build the CLEAN base model
#   first, then for each of ITS keys we find the matching trained tensor by suffix.
SAVE_PATH = "/kaggle/working/donut_finetuned_v2"
import torch, shutil, os
from transformers import VisionEncoderDecoderModel

# CRITICAL: wipe any stale checkpoint. save_pretrained does NOT delete old files,
# so a leftover model.safetensors.index.json from an earlier (broken, sharded, peft)
# save would take precedence and reload the OLD 57525-vocab peft weights.
shutil.rmtree(SAVE_PATH, ignore_errors=True)
os.makedirs(SAVE_PATH, exist_ok=True)

tok_len = len(processor.tokenizer)
LORA_ALPHA, LORA_R = 32, 16          # must match the LoraConfig in cell 3.3
scaling = LORA_ALPHA / LORA_R

sd = model.state_dict()

# 0) Fold any still-separate LoRA deltas into their base_layer weights IN the source
#    dict, so the merged weight is what we copy over. (folded=0 if already merged.)
folded = 0
merged_src = dict(sd)
for k in list(sd.keys()):
    if not k.endswith("lora_A.default.weight"):
        continue
    pfx = k[:-len("lora_A.default.weight")]           # e.g. ...q_proj.
    A = sd[pfx + "lora_A.default.weight"]
    B = sd[pfx + "lora_B.default.weight"]
    base_k = pfx + "base_layer.weight"
    if base_k in merged_src and merged_src[base_k].shape == (B.shape[0], A.shape[1]):
        merged_src[base_k] = merged_src[base_k] + (B @ A).to(merged_src[base_k].dtype) * scaling
        folded += 1
print("folded LoRA layers:", folded)

# 1) Normalize source keys -> canonical HF names, dropping peft-only entries.
def _canon(k):
    if k.startswith("base_model.model."):
        k = k[len("base_model.model."):]
    return k.replace(".base_layer", "").replace(".modules_to_save.default", "")

canon_src = {}
for k, v in merged_src.items():
    if "lora_" in k or "original_module" in k:
        continue
    canon_src[_canon(k)] = v

# 2) Build the clean model and fill each of ITS params. Exact-name match first, then
#    a unique-suffix fallback so naming quirks can't silently drop trained weights.
clean = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
clean.decoder.resize_token_embeddings(tok_len)
clean_sd = clean.state_dict()

# suffix index of source keys for fallback matching
from collections import defaultdict
by_suffix = defaultdict(list)
for sk in canon_src:
    by_suffix[sk.split(".")[-3:][0] + "." + ".".join(sk.split(".")[-2:])].append(sk)

new_sd, matched, shape_skip, unmatched = {}, 0, 0, 0
for ck, cv in clean_sd.items():
    src = None
    if ck in canon_src:
        src = canon_src[ck]
    else:
        cand = [s for s in canon_src if s.endswith(ck) or ck.endswith(s)]
        if len(cand) == 1:
            src = canon_src[cand[0]]
    if src is None:
        unmatched += 1
        new_sd[ck] = cv                 # keep base weight
        continue
    if src.shape != cv.shape:
        shape_skip += 1
        new_sd[ck] = cv                 # keep base weight (shape mismatch)
        continue
    new_sd[ck] = src
    matched += 1

print(f"matched: {matched} | shape_skipped: {shape_skip} | unmatched: {unmatched} | total: {len(clean_sd)}")
missing, unexpected = clean.load_state_dict(new_sd, strict=False)
print("load_state_dict -> missing:", len(missing), "| unexpected:", len(unexpected))

# 2b) Force the trained enlarged embed + lm_head explicitly (the tags live here).
with torch.no_grad():
    emb_src = canon_src.get("decoder.model.decoder.embed_tokens.weight")
    lm_src  = canon_src.get("decoder.lm_head.weight")
    if emb_src is not None and emb_src.shape[0] == tok_len:
        clean.decoder.get_input_embeddings().weight.copy_(emb_src)
    if lm_src is not None and lm_src.shape[0] == tok_len:
        clean.decoder.get_output_embeddings().weight.copy_(lm_src)

# 3) Guard: if almost nothing matched, STOP — the saved model would be untrained.
assert matched > 0.5 * len(clean_sd), (
    f"only {matched}/{len(clean_sd)} tensors matched -> key mapping is wrong, NOT saving. "
    f"Sample clean keys: {list(clean_sd)[:3]} | sample src keys: {list(canon_src)[:3]}"
)

# 4) Hard sanity checks on vocab.
assert clean.decoder.get_input_embeddings().weight.shape[0]  == tok_len, "embed vocab mismatch!"
assert clean.decoder.get_output_embeddings().weight.shape[0] == tok_len, "lm_head vocab mismatch!"

clean.config.tie_word_embeddings = False
clean.config.vocab_size = tok_len
clean.decoder.config.vocab_size = tok_len
clean.config.pad_token_id = processor.tokenizer.pad_token_id
clean.config.decoder_start_token_id = processor.tokenizer.bos_token_id
if clean.config.decoder_start_token_id is None:
    clean.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s>")

clean.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)
print(f"Clean plain model saved to: {SAVE_PATH} (vocab={tok_len}, matched={matched}, folded={folded})")

# 6) VERIFY ON DISK: reload from scratch and confirm no peft keys + right vocab.
#    This is the ground truth — it's exactly what predict_donut.py will load.
import glob
print("\nfiles on disk:", [os.path.basename(p) for p in glob.glob(SAVE_PATH + "/*")])
_reloaded = VisionEncoderDecoderModel.from_pretrained(SAVE_PATH)
_emb_rows = _reloaded.decoder.get_input_embeddings().weight.shape[0]
_peft_keys = [k for k in _reloaded.state_dict() if "lora_" in k or "modules_to_save" in k]
print(f"RELOAD CHECK -> embed rows: {_emb_rows} (want {tok_len}) | peft keys left: {len(_peft_keys)}")
assert _emb_rows == tok_len, f"DISK MODEL WRONG VOCAB {_emb_rows} != {tok_len}"
assert len(_peft_keys) == 0, f"DISK MODEL STILL HAS PEFT KEYS: {_peft_keys[:3]}"
print("VERIFIED: on-disk model is clean plain HF with correct vocab. Safe to run inference.")
del _reloaded


In [ ]:
# 3.6b-backup: upload the trained model to HF so a session timeout never loses it.
# HARD GUARD: before uploading, reload the ON-DISK model and refuse if it is a PEFT
# checkpoint or has the wrong vocab. This is exactly what got corrupted last time —
# a PEFT model (missing modules_to_save embed/lm_head) was uploaded, so on restore the
# tag embeddings were random -> CUDA index assert / 0.000 F1. Never upload that again.
from huggingface_hub import HfApi
from transformers import DonutProcessor, VisionEncoderDecoderModel

def _verify_clean(path):
    tok_len = len(DonutProcessor.from_pretrained(path).tokenizer)
    m = VisionEncoderDecoderModel.from_pretrained(path)
    peft_keys = [k for k in m.state_dict() if "lora_" in k or "modules_to_save" in k]
    emb_rows = m.decoder.get_input_embeddings().weight.shape[0]
    del m
    assert len(peft_keys) == 0, f"REFUSING UPLOAD: on-disk model still has PEFT keys: {peft_keys[:3]}"
    assert emb_rows == tok_len, f"REFUSING UPLOAD: embed rows {emb_rows} != tokenizer {tok_len}"
    return emb_rows, tok_len

try:
    emb_rows, tok_len = _verify_clean(SAVE_PATH)
    print(f"PRE-UPLOAD CHECK OK -> clean plain model, vocab={emb_rows}, 0 peft keys. Uploading...")
    api = HfApi()
    api.create_repo(MODEL_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(
        folder_path=SAVE_PATH,
        repo_id=MODEL_REPO,
        repo_type="model",
    )
    print(f"Model backed up to https://huggingface.co/{MODEL_REPO}")
    print("Restore later with: huggingface_hub.snapshot_download(repo_id=MODEL_REPO)")
except AssertionError as e:
    print(e)
    print("-> Re-run cell 3.6 (clean save) until it prints VERIFIED, THEN re-run this cell.")
except Exception as e:
    print("Model backup skipped/failed (check HF_TOKEN):", e)


## 4. Modify Code for Testing

The original run scored **0.000 F1** on every tier (`mostly_missing: 20`). Two competing
hypotheses:

1. **Trained weights are fine**, but `predict_donut.py` (fresh process) never re-applies
   `pad_token_id` / `decoder_start_token_id`, so generation breaks.
2. **Save/restore of `modules_to_save` (embed_tokens / lm_head)** didn't round-trip, so the
   custom-tag embeddings are random at inference time.

**Cell 4.1 disambiguates them:** it generates *in-session* using the just-trained model.
If real tags appear here, hypothesis #1 is correct and the loader fix (4.2) resolves it.

In [ ]:
# 4.1 IN-SESSION SMOKE TEST — run right after training (model still in memory)
# Proves whether the trained weights themselves generate valid tags.
sample = train_dataset[0]
pixel_values = sample["pixel_values"].unsqueeze(0).to(model.device)
decoder_input_ids = processor.tokenizer(
    "<s_extraction>", add_special_tokens=False, return_tensors="pt"
).input_ids.to(model.device)

with torch.no_grad():
    out = model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_new_tokens=200,
        eos_token_id=processor.tokenizer.convert_tokens_to_ids("</s_extraction>"),
    )

insession_decoded = processor.tokenizer.batch_decode(out)[0]
print(insession_decoded)

# Quick diagnosis flag used later by the assertions cell
insession_has_tags = any(f"<s_{f}>" in insession_decoded for f in HEADER_FIELDS)
print("\n>>> in-session produced real field tags:", insession_has_tags)


In [ ]:
# 4.1b CHECK C — teacher-forced PRED vs GOLD (no generation, pure forward pass)
# Run right after training (model still in memory), or after restoring the model.
# Decides: did the weights actually LEARN, or is only free-running generate() broken?
import torch

b = val_dataset[0]
pv = b["pixel_values"].unsqueeze(0).to(model.device)
labels = b["labels"].unsqueeze(0).to(model.device)

with torch.no_grad():
    logits = model(pixel_values=pv, labels=labels).logits

pred = logits.argmax(-1)[0]
gold = labels[0].clone()
gold[gold == -100] = processor.tokenizer.pad_token_id

tok = processor.tokenizer
print("PRED:", tok.decode(pred[:40]))
print("GOLD:", tok.decode(gold[:40]))

# Interpretation:
#   PRED ~= GOLD (field tags line up)  -> weights learned; only generate() is broken (cheap fix)
#   PRED = garbage (mark/repeat)       -> training didn't learn; fix the training run itself


In [ ]:
# 4.2 FIXED predict_donut.py — re-applies pad / decoder_start / eos in the fresh process
%%writefile src/models/predict_donut.py
"""Run the fine-tuned Donut model on the test set and save predictions in the
SAME format as zero_shot.py so scorer.py works unchanged.

FIX vs original: a fresh process reloads base donut-base and does NOT inherit the
config set during training. We re-apply pad_token_id + decoder_start_token_id on
the base model AND mirror them (plus the custom eos tag) onto generation_config,
so generation starts and stops on the correct custom tags.
"""
import argparse
import csv
import json
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import torch
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel

from src.data.donut_format import decode_donut_output, HEADER_FIELDS, TASK_START_TOKEN, TASK_END_TOKEN

TEST_CSV = Path("data/splits/test.csv")
RUNS_ROOT = Path("results/runs")


def load_finetuned_model(checkpoint_dir: str, base_model_name: str = "naver-clova-ix/donut-base"):
    processor = DonutProcessor.from_pretrained(checkpoint_dir)

    # checkpoint_dir is a MERGED full model (LoRA already folded in at save time),
    # so we load it directly — no base reload, no PeftModel, no adapter key remap.
    model = VisionEncoderDecoderModel.from_pretrained(checkpoint_dir)

    # Safety net: if the saved embedding table doesn't match the tokenizer vocab, resize.
    # A mismatch here is exactly what caused the CUDA "srcIndex < srcSelectDimSize" asserts.
    tok_len = len(processor.tokenizer)
    if model.decoder.get_input_embeddings().weight.shape[0] != tok_len:
        print(f"WARNING: embed rows {model.decoder.get_input_embeddings().weight.shape[0]} "
              f"!= tokenizer {tok_len} -> resizing")
        model.decoder.resize_token_embeddings(tok_len)

    # --- re-apply config the fresh process may not have inherited ---
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.decoder_start_token_id = processor.tokenizer.bos_token_id
    if model.config.decoder_start_token_id is None:
        model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s>")

    # Mirror onto generation_config so generate() uses the right start/stop/pad
    model.generation_config.eos_token_id = processor.tokenizer.convert_tokens_to_ids(TASK_END_TOKEN)
    model.generation_config.decoder_start_token_id = model.config.decoder_start_token_id
    model.generation_config.pad_token_id = model.config.pad_token_id
    # --------------------------------------------------------------------------

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    return model, processor, device


def generate_prediction(model, processor, device, image_path: str, max_new_tokens: int = 512) -> str:
    """GUIDED (constrained) decoding.

    Free-running generation collapses/scrambles the tag structure: the model
    over-learned <s_extraction> and emits doubled/out-of-order tags, so the parser
    gets garbage (F1 ~0.009). BUT teacher-forcing proved the model reads the VALUES
    off the image correctly. So instead of asking it to emit the whole
    <s_field>value</s_field> skeleton itself, we SUPPLY the skeleton and let it fill
    in ONLY the value for one field at a time:

      seq = [decoder_start, <s_extraction>]
      for field:
          seq += [<s_field>]
          generate until </s_field>, with every OTHER tag banned
          append the produced value tokens + </s_field>

    Each field decodes greedily (num_beams=1) with all structural tags except its own
    closing tag forbidden, so the value can't wander into another field. We then
    reconstruct a clean, correctly-ordered Donut string that decode_donut_output()
    parses trivially.
    """
    tok = processor.tokenizer
    image = Image.open(image_path).convert("RGB")
    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

    dsi = model.config.decoder_start_token_id
    start_id = tok.convert_tokens_to_ids(TASK_START_TOKEN)
    end_id = tok.convert_tokens_to_ids(TASK_END_TOKEN)

    # All structural tag ids (both open + close for every field, plus extraction tags).
    all_tag_ids = {start_id, end_id}
    for f in HEADER_FIELDS:
        all_tag_ids.add(tok.convert_tokens_to_ids(f"<s_{f}>"))
        all_tag_ids.add(tok.convert_tokens_to_ids(f"</s_{f}>"))

    seq = [dsi, start_id]
    parts = []  # reconstructed clean donut string pieces
    for f in HEADER_FIELDS:
        open_id = tok.convert_tokens_to_ids(f"<s_{f}>")
        close_id = tok.convert_tokens_to_ids(f"</s_{f}>")
        seq_field = seq + [open_id]

        # Forbid every structural tag EXCEPT this field's own closing tag (the eos),
        # so the model can only emit value tokens then stop on </s_field>.
        banned = [[t] for t in all_tag_ids if t != close_id]

        dec_in = torch.tensor([seq_field], device=device)
        with torch.no_grad():
            out = model.generate(
                pixel_values,
                decoder_input_ids=dec_in,
                max_new_tokens=48,                 # values are short; keeps it fast
                pad_token_id=tok.pad_token_id,
                eos_token_id=close_id,             # stop at THIS field's close tag
                bad_words_ids=banned,
                no_repeat_ngram_size=3,            # guard against digit/word loops
                num_beams=1,                       # greedy value read (matches teacher-forcing)
                return_dict_in_generate=True,
            )
        gen = out.sequences[0].tolist()[len(seq_field):]
        value_ids = [t for t in gen if t not in all_tag_ids and t != tok.pad_token_id]
        value = tok.decode(value_ids, skip_special_tokens=True).strip()

        parts.append(f"<s_{f}>{value}</s_{f}>")
        # advance the running context with the value + close tag for the next field
        seq = seq_field + value_ids + [close_id]

    return f"{TASK_START_TOKEN}{''.join(parts)}{TASK_END_TOKEN}"


def load_test_rows(limit: int = None):
    with open(TEST_CSV, "r", newline="") as f:
        rows = list(csv.DictReader(f))
    return rows[:limit] if limit else rows


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--limit", type=int, default=None)
    parser.add_argument("--run-name", type=str, default="donut_finetuned")
    args = parser.parse_args()

    print(f"Loading fine-tuned model from: {args.checkpoint}")
    model, processor, device = load_finetuned_model(args.checkpoint)
    print(f"Model loaded on: {device}")
    print(f"gen eos={model.generation_config.eos_token_id} "
          f"start={model.generation_config.decoder_start_token_id} "
          f"pad={model.generation_config.pad_token_id}\n")

    rows = load_test_rows(limit=args.limit)
    print(f"Running inference on {len(rows)} test images...\n")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_id = f"{timestamp}_{args.run_name}"
    run_dir = RUNS_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    results_path = run_dir / "zero_shot_results.jsonl"
    config_path = run_dir / "run_config.json"

    results = []
    for i, row in enumerate(rows, 1):
        try:
            raw_sequence = generate_prediction(model, processor, device, row["image_path"])
            decoded = decode_donut_output(raw_sequence, HEADER_FIELDS)
            prediction_raw = json.dumps(decoded)
            error = None
        except Exception as e:
            print(f"  [{i}/{len(rows)}] FAILED: {row['doc_id']} ({row['level']}) - {e}")
            prediction_raw = None
            error = str(e)

        results.append({
            "doc_id": row["doc_id"],
            "template_type": row["template_type"],
            "level": row["level"],
            "image_path": row["image_path"],
            "label_path": row["label_path"],
            "prediction_raw": prediction_raw,
            "error": error,
        })
        if error is None:
            print(f"  [{i}/{len(rows)}] done: {row['doc_id']} ({row['level']})")

    with open(results_path, "w") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")

    n_failed = sum(1 for r in results if r["error"] is not None)
    by_template, by_level = defaultdict(int), defaultdict(int)
    for r in results:
        by_template[r["template_type"]] += 1
        by_level[r["level"]] += 1

    run_config = {
        "run_id": run_id,
        "timestamp": datetime.now().isoformat(),
        "model": "donut-base + LoRA (fine-tuned)",
        "checkpoint": args.checkpoint,
        "total_images": len(results),
        "succeeded": len(results) - n_failed,
        "failed": n_failed,
        "coverage_by_template": dict(by_template),
        "coverage_by_level": dict(by_level),
        "results_file": str(results_path).replace("\\", "/"),
    }
    with open(config_path, "w") as f:
        json.dump(run_config, f, indent=2)

    print(f"\nDone. {len(results) - n_failed}/{len(results)} succeeded.")
    print(f"Run ID: {run_id}")
    print(f"\nNext step: python -m src.eval.scorer --run-id {run_id}")


if __name__ == "__main__":
    main()


## 5. Run and Compare Results

Run inference with the **fixed** script on a 20-image slice, then score it. Compare against
the original 0.000-F1 baseline — F1 > 0 confirms the fix worked.

In [ ]:
# 5.0 (fresh session only) Restore the trained model from HF — skip if you just trained.
# Lets you run inference/scoring without retraining. Needs the 2.2b-config cell run first.
import os
from huggingface_hub import snapshot_download

SAVE_PATH = "/kaggle/working/donut_finetuned_v2"
if not os.path.isdir(SAVE_PATH):
    try:
        snapshot_download(repo_id=MODEL_REPO, repo_type="model", local_dir=SAVE_PATH)
        print(f"Model restored from HF to {SAVE_PATH}")
    except Exception as e:
        print("Could not restore model from HF (train first or check MODEL_REPO):", e)
else:
    print(f"Model already present at {SAVE_PATH} — using local copy.")


In [ ]:
# 5.0b VERIFY the restored checkpoint is the CLEAN model — not the old corrupt PEFT one.
# Your error (`embed rows 57525 != tokenizer 57547`, `...lora_*/modules_to_save... MISSING`,
# `Old embeddings are of type ModulesToSaveWrapper`) means the folder holds a PEFT/base
# checkpoint whose trained weights are GONE. This cell detects that and tries to recover.
import os, shutil
from transformers import DonutProcessor, VisionEncoderDecoderModel

SAVE_PATH = "/kaggle/working/donut_finetuned_v2"

def _inspect(path):
    """Return (is_clean, emb_rows, tok_len, n_peft_keys) for a checkpoint on disk."""
    tok_len = len(DonutProcessor.from_pretrained(path).tokenizer)
    m = VisionEncoderDecoderModel.from_pretrained(path)
    peft_keys = [k for k in m.state_dict() if "lora_" in k or "modules_to_save" in k]
    emb_rows = m.decoder.get_input_embeddings().weight.shape[0]
    del m
    is_clean = (len(peft_keys) == 0) and (emb_rows == tok_len)
    return is_clean, emb_rows, tok_len, len(peft_keys)

def _try_download():
    from huggingface_hub import snapshot_download
    shutil.rmtree(SAVE_PATH, ignore_errors=True)
    snapshot_download(repo_id=MODEL_REPO, repo_type="model", local_dir=SAVE_PATH)

status = "unknown"
if os.path.isdir(SAVE_PATH):
    try:
        clean, emb, tok, npeft = _inspect(SAVE_PATH)
        print(f"LOCAL checkpoint -> embed_rows={emb} tokenizer={tok} peft_keys={npeft} clean={clean}")
        if not clean:
            print("LOCAL is CORRUPT (base/PEFT). Deleting and re-pulling from HF...")
            _try_download()
            clean, emb, tok, npeft = _inspect(SAVE_PATH)
            print(f"HF checkpoint    -> embed_rows={emb} tokenizer={tok} peft_keys={npeft} clean={clean}")
        status = "clean" if clean else "corrupt"
    except Exception as e:
        print("Inspection failed:", e); status = "corrupt"
else:
    try:
        print("No local checkpoint. Pulling from HF...")
        _try_download()
        clean, emb, tok, npeft = _inspect(SAVE_PATH)
        print(f"HF checkpoint -> embed_rows={emb} tokenizer={tok} peft_keys={npeft} clean={clean}")
        status = "clean" if clean else "corrupt"
    except Exception as e:
        print("Could not pull from HF:", e); status = "missing"

print("\n" + "=" * 64)
if status == "clean":
    print("OK: checkpoint is the CLEAN plain model. Proceed to 4.2 (write script) -> 5.1.")
else:
    print("PROBLEM: no clean fine-tuned model is available (HF copy is the OLD corrupt one).")
    print("The trained weights are NOT recoverable from this checkpoint. You must RE-TRAIN:")
    print("  1) run the data cells (2.1, 2.2, 2.2b-config, 2.2b-restore, 3.1)")
    print("  2) run 2.5 -> 2.6 -> 3.3 -> 3.4 -> 3.5 (train)")
    print("  3) run 3.6 (CLEAN save — must print 'VERIFIED') then 3.6b (re-upload clean model)")
    print("  4) THEN come back to 4.2 -> 5.1. The 3.6b guard prevents re-uploading a bad model.")
print("=" * 64)


In [ ]:
# 5.0c RECOVERY — what did you ACTUALLY upload? Maybe the weights aren't lost, just saved
# in the wrong FORMAT. If you saved a PEFT adapter (adapter_model.safetensors) or a base
# save, loading it as a PLAIN VisionEncoderDecoderModel shows everything MISSING even though
# the trained tensors ARE in the file. This cell inspects the real files and tries to
# recover the trained model WITHOUT retraining.
import os
from huggingface_hub import list_repo_files

# 1) What files are really in the HF model repo?
try:
    files = list_repo_files(repo_id=MODEL_REPO, repo_type="model")
    print(f"Files in HF repo {MODEL_REPO}:")
    for f in sorted(files):
        print("   ", f)
except Exception as e:
    files = []
    print("Could not list HF repo files:", e)

has_adapter = any("adapter_model" in f or "adapter_config" in f for f in files)
has_full    = any(f.endswith("model.safetensors") or f.endswith("pytorch_model.bin")
                  or "model.safetensors.index.json" in f for f in files)
print(f"\n-> adapter files present: {has_adapter} | full-model files present: {has_full}")

RECOVERED = False
if has_adapter:
    # 2) RECOVER via PEFT: load base donut + custom tags, resize, then apply the adapter.
    #    The adapter's modules_to_save carries your trained embed_tokens + lm_head.
    print("\nADAPTER found -> attempting PEFT recovery (no retrain needed)...")
    import torch
    from transformers import DonutProcessor, VisionEncoderDecoderModel
    from peft import PeftModel
    from huggingface_hub import snapshot_download

    adapter_dir = snapshot_download(repo_id=MODEL_REPO, repo_type="model")
    MODEL_NAME = "naver-clova-ix/donut-base"

    proc = DonutProcessor.from_pretrained(adapter_dir) if any("tokenizer" in f for f in files) \
           else DonutProcessor.from_pretrained(MODEL_NAME)
    # rebuild the SAME 22 custom tags the training used, so vocab matches the adapter
    HEADER_FIELDS = [
        "account_number","ifsc_code","micr_code","bank_name","account_holder",
        "opening_balance","closing_balance","start_date","end_date","statement_date",
    ]
    if len(proc.tokenizer) == 57525:  # tags not in this tokenizer -> add them
        toks = ["<s_extraction>","</s_extraction>"]
        for f in HEADER_FIELDS:
            toks += [f"<s_{f}>", f"</s_{f}>"]
        proc.tokenizer.add_tokens(toks)

    base = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
    base.decoder.resize_token_embeddings(len(proc.tokenizer))
    try:
        peft_model = PeftModel.from_pretrained(base, adapter_dir)
        merged = peft_model.merge_and_unload()      # fold LoRA + modules_to_save into base
        emb_rows = merged.decoder.get_input_embeddings().weight.shape[0]
        print(f"PEFT recovery OK -> merged embed rows: {emb_rows} (tokenizer {len(proc.tokenizer)})")

        # 3) Re-save as the CLEAN plain model the pipeline expects, then re-upload.
        import shutil
        shutil.rmtree(SAVE_PATH, ignore_errors=True)
        merged.config.tie_word_embeddings = False
        merged.config.vocab_size = len(proc.tokenizer)
        merged.decoder.config.vocab_size = len(proc.tokenizer)
        merged.config.pad_token_id = proc.tokenizer.pad_token_id
        merged.config.decoder_start_token_id = proc.tokenizer.bos_token_id or \
            proc.tokenizer.convert_tokens_to_ids("<s>")
        merged.save_pretrained(SAVE_PATH)
        proc.save_pretrained(SAVE_PATH)
        # verify clean
        chk = VisionEncoderDecoderModel.from_pretrained(SAVE_PATH)
        pk = [k for k in chk.state_dict() if "lora_" in k or "modules_to_save" in k]
        print(f"Re-saved CLEAN model -> peft_keys={len(pk)} embed_rows="
              f"{chk.decoder.get_input_embeddings().weight.shape[0]}")
        RECOVERED = (len(pk) == 0 and chk.decoder.get_input_embeddings().weight.shape[0] == len(proc.tokenizer))
        del chk
    except Exception as e:
        print("PEFT recovery failed:", e)

print("\n" + "=" * 64)
if RECOVERED:
    print("RECOVERED: trained model rebuilt from the adapter and saved CLEAN to SAVE_PATH.")
    print("NEXT: run 3.6b to re-upload the CLEAN model (replaces the bad one), then 4.2 -> 5.1.")
elif has_full and not has_adapter:
    print("HF repo has a FULL model.safetensors but it is the BASE (57525) — the clean-save")
    print("(3.6) never captured your trained tags, and no adapter was uploaded either.")
    print("-> The trained weights are NOT on HF. You must RE-TRAIN (3.3->3.6->3.6b).")
else:
    print("No recoverable adapter found. If you still have the training kernel alive, run 3.6")
    print("NOW to save the in-memory model. Otherwise RE-TRAIN (3.3->3.6->3.6b).")
print("=" * 64)


In [ ]:
# 5.0d RECOVERY (robust) — rebuild the CLEAN model straight from the on-disk PEFT
# `model.safetensors`, WITHOUT PeftModel.merge_and_unload (which failed on the
# modules_to_save key remap). Your 813MB model.safetensors holds the FULL PEFT state:
#   - base weights
#   - lora_A/lora_B for q/v proj
#   - modules_to_save.default.weight for embed_tokens + lm_head  <-- your trained tags (57547)
# This is EXACTLY what cell 3.6 folds in-memory; here we do the same folding from DISK.
import os, glob, shutil
import torch
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import DonutProcessor, VisionEncoderDecoderModel

MODEL_NAME = "naver-clova-ix/donut-base"
SAVE_PATH  = "/kaggle/working/donut_finetuned_v2"

# 1) Get the repo files locally and locate the FULL peft state dict (model.safetensors).
ckpt_dir = snapshot_download(repo_id=MODEL_REPO, repo_type="model")
proc = DonutProcessor.from_pretrained(ckpt_dir)
tok_len = len(proc.tokenizer)
print("tokenizer length:", tok_len)

st_files = [p for p in glob.glob(os.path.join(ckpt_dir, "*.safetensors"))
            if os.path.basename(p) != "adapter_model.safetensors"]
assert st_files, "no full model.safetensors found in the repo"
sd = {}
for p in st_files:
    sd.update(load_file(p))
print("loaded state dict tensors:", len(sd))
n_peft = sum(1 for k in sd if "lora_" in k or "modules_to_save" in k)
print("peft-style keys in file:", n_peft)

LORA_ALPHA, LORA_R = 32, 16          # must match LoraConfig in cell 3.3
scaling = LORA_ALPHA / LORA_R

# 2) Fold LoRA deltas into their base_layer weights.
folded = 0
merged_src = dict(sd)
for k in list(sd.keys()):
    if not k.endswith("lora_A.default.weight"):
        continue
    pfx = k[:-len("lora_A.default.weight")]
    A, B = sd.get(pfx + "lora_A.default.weight"), sd.get(pfx + "lora_B.default.weight")
    base_k = pfx + "base_layer.weight"
    if A is not None and B is not None and base_k in merged_src \
       and merged_src[base_k].shape == (B.shape[0], A.shape[1]):
        merged_src[base_k] = merged_src[base_k] + (B @ A).to(merged_src[base_k].dtype) * scaling
        folded += 1
print("folded LoRA layers:", folded)

# 3) Canonicalize keys -> plain HF names; prefer modules_to_save (trained) over original_module.
def _canon(k):
    if k.startswith("base_model.model."):
        k = k[len("base_model.model."):]
    return k.replace(".base_layer", "").replace(".modules_to_save.default", "")

canon_src = {}
for k, v in merged_src.items():
    if "lora_" in k:
        continue
    if "original_module" in k:          # the FROZEN copy — skip, we want modules_to_save
        continue
    canon_src[_canon(k)] = v
print("canonical source keys:", len(canon_src))

# 4) Build clean base, resize, fill each of its params by exact then unique-suffix match.
clean = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
clean.decoder.resize_token_embeddings(tok_len)
clean_sd = clean.state_dict()

new_sd, matched, shape_skip, unmatched = {}, 0, 0, 0
for ck, cv in clean_sd.items():
    src = canon_src.get(ck)
    if src is None:
        cand = [s for s in canon_src if s.endswith(ck) or ck.endswith(s)]
        if len(cand) == 1:
            src = canon_src[cand[0]]
    if src is None:
        unmatched += 1; new_sd[ck] = cv; continue
    if src.shape != cv.shape:
        shape_skip += 1; new_sd[ck] = cv; continue
    new_sd[ck] = src; matched += 1
print(f"matched={matched} shape_skipped={shape_skip} unmatched={unmatched} total={len(clean_sd)}")
clean.load_state_dict(new_sd, strict=False)

# 4b) Force the trained enlarged embed + lm_head explicitly (the tags live here).
with torch.no_grad():
    emb_src = canon_src.get("decoder.model.decoder.embed_tokens.weight")
    lm_src  = canon_src.get("decoder.lm_head.weight")
    if emb_src is not None and emb_src.shape[0] == tok_len:
        clean.decoder.get_input_embeddings().weight.copy_(emb_src); print("forced embed_tokens 57547")
    if lm_src is not None and lm_src.shape[0] == tok_len:
        clean.decoder.get_output_embeddings().weight.copy_(lm_src); print("forced lm_head 57547")

assert matched > 0.5 * len(clean_sd), f"only {matched}/{len(clean_sd)} matched -> mapping wrong"
assert clean.decoder.get_input_embeddings().weight.shape[0] == tok_len, "embed vocab mismatch"

# 5) Config + save + verify clean on disk.
clean.config.tie_word_embeddings = False
clean.config.vocab_size = tok_len
clean.decoder.config.vocab_size = tok_len
clean.config.pad_token_id = proc.tokenizer.pad_token_id
clean.config.decoder_start_token_id = proc.tokenizer.bos_token_id or \
    proc.tokenizer.convert_tokens_to_ids("<s>")

shutil.rmtree(SAVE_PATH, ignore_errors=True)
clean.save_pretrained(SAVE_PATH)
proc.save_pretrained(SAVE_PATH)

_r = VisionEncoderDecoderModel.from_pretrained(SAVE_PATH)
_pk = [k for k in _r.state_dict() if "lora_" in k or "modules_to_save" in k]
_emb = _r.decoder.get_input_embeddings().weight.shape[0]
del _r
print(f"\nRELOAD CHECK -> embed_rows={_emb} (want {tok_len}) | peft_keys={len(_pk)}")
RECOVERED = (_emb == tok_len and len(_pk) == 0)

print("\n" + "=" * 64)
if RECOVERED:
    print("RECOVERED (no retrain): clean plain model rebuilt from model.safetensors and saved.")
    print("NEXT: run 4.1b/5.1b teacher-force to confirm PRED~=GOLD, then 3.6b to re-upload,")
    print("      then 4.2 -> 5.1. If PRED is garbage, the trained tags didn't fold -> retrain.")
else:
    print("Reconstruction did not verify clean. Fall back to RE-TRAIN (3.3->3.6->3.6b).")
print("=" * 64)


In [ ]:
# 5.1 Inference with the fixed script (20-image slice for a quick check)
!python -m src.models.predict_donut --checkpoint /kaggle/working/donut_finetuned_v2 --limit 20
# For a TRUSTWORTHY F1, run the full test set (drop --limit) — 20 images is a tiny, noisy sample:
# !python -m src.models.predict_donut --checkpoint /kaggle/working/donut_finetuned_v2

In [ ]:
# 5.1b DEBUG — why is generate() empty though CHECK C matched?
# Loads the ON-DISK model exactly like predict_donut.py, generates on ONE test image,
# and prints the RAW string + token ids. This tells us WHICH of two things is happening:
#   (A) generate emits </s_extraction> immediately -> raw is basically just start+end (gen collapse)
#   (B) raw has real tags but decode_donut_output() fails to parse them (parser mismatch)
import csv, torch
from pathlib import Path
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel
from src.data.donut_format import decode_donut_output, HEADER_FIELDS, TASK_START_TOKEN, TASK_END_TOKEN

CKPT = "/kaggle/working/donut_finetuned_v2"
proc = DonutProcessor.from_pretrained(CKPT)
mdl  = VisionEncoderDecoderModel.from_pretrained(CKPT)
mdl.config.pad_token_id = proc.tokenizer.pad_token_id
mdl.config.decoder_start_token_id = proc.tokenizer.bos_token_id
if mdl.config.decoder_start_token_id is None:
    mdl.config.decoder_start_token_id = proc.tokenizer.convert_tokens_to_ids("<s>")
dev = "cuda" if torch.cuda.is_available() else "cpu"
mdl.to(dev).eval()

# grab first test image
with open("data/splits/test.csv", newline="") as f:
    row0 = next(csv.DictReader(f))
img = Image.open(row0["image_path"]).convert("RGB")
pv = proc(img, return_tensors="pt").pixel_values.to(dev)

# CRITICAL FIX: training shifts labels right and PREPENDS decoder_start_token_id (0),
# so the decoder always saw [0, <s_extraction>, <field tags>...]. If we start generation
# from [<s_extraction>] alone (no leading 0), the model sees a position-0 context it never
# trained on and collapses (repeats <s_extraction> forever). So prepend decoder_start id.
dsi = mdl.config.decoder_start_token_id
prompt_ids = proc.tokenizer(TASK_START_TOKEN, add_special_tokens=False, return_tensors="pt").input_ids
dec_in = torch.cat([torch.tensor([[dsi]]), prompt_ids], dim=1).to(dev)

eos_id  = proc.tokenizer.convert_tokens_to_ids(TASK_END_TOKEN)
start_id = proc.tokenizer.convert_tokens_to_ids(TASK_START_TOKEN)
print(f"TASK_START={TASK_START_TOKEN}({start_id})  TASK_END={TASK_END_TOKEN}({eos_id})  "
      f"decoder_start={mdl.config.decoder_start_token_id}  pad={mdl.config.pad_token_id}")
print("decoder_input_ids (with prepended start):", dec_in[0].tolist())

with torch.no_grad():
    out = mdl.generate(
        pv, decoder_input_ids=dec_in, max_new_tokens=512,
        pad_token_id=proc.tokenizer.pad_token_id, eos_token_id=eos_id,
        return_dict_in_generate=True,
    )
seq = out.sequences[0]
raw = proc.tokenizer.decode(seq, skip_special_tokens=False)
print("\nnum tokens generated:", seq.shape[0])
print("first 30 token ids:", seq[:30].tolist())
print("\nRAW (special tokens KEPT):\n", raw[:800])
print("\nParsed by decode_donut_output ->", decode_donut_output(raw, HEADER_FIELDS))

# ============================================================================
# DECISIVE TEST: teacher-force the RELOADED DISK MODEL (mdl) on a val sample.
# CHECK C (4.1b) passed on the IN-MEMORY model. If the DISK model also matches
# GOLD here, the weights saved fine and the bug is purely generation config.
# If the DISK model produces garbage here too, the SAVE was lossy (likely lm_head)
# and we must fix the save (or save the in-memory model a different way).
# ============================================================================
print("\n" + "=" * 60)
print("TEACHER-FORCE on RELOADED DISK MODEL (mdl):")
vb = val_dataset[0]
vpv = vb["pixel_values"].unsqueeze(0).to(dev)
vlabels = vb["labels"].unsqueeze(0).to(dev)
with torch.no_grad():
    vlogits = mdl(pixel_values=vpv, labels=vlabels).logits
vpred = vlogits.argmax(-1)[0]
vgold = vlabels[0].clone()
vgold[vgold == -100] = proc.tokenizer.pad_token_id
print("DISK PRED:", proc.tokenizer.decode(vpred[:40]))
print("GOLD     :", proc.tokenizer.decode(vgold[:40]))
_match = (vpred[:40] == vgold[:40]).float().mean().item()
print(f"token match (first 40): {_match:.2f}")
print("If match ~0 -> the SAVED model is corrupt (lm_head/embed lost). "
      "If match high -> weights fine, generation-config bug only.")

# ============================================================================
# THE FIX: the model over-learned <s_extraction> and re-emits it every step,
# collapsing free generation. But <s_extraction> should appear ONCE (at start).
# Forbid it after the prompt via bad_words_ids -> the model is forced to the
# next-best token, which teacher-forcing proved is the real field sequence.
# ============================================================================
print("\n" + "=" * 60)
print("GUIDED (constrained) DECODING — supply the tag skeleton, fill one value at a time:")
# Free-gen scrambles the tags (doubled/out-of-order) even with penalties, because the
# model's control over structure is weak (teacher-force match ~0.65). But it READS values
# fine. So we feed [<s_field>], let it generate the value greedily until </s_field> with
# every OTHER tag banned, then move to the next field. This is what predict_donut.py does.
all_tag_ids = {start_id, eos_id}
for f in HEADER_FIELDS:
    all_tag_ids.add(proc.tokenizer.convert_tokens_to_ids(f"<s_{f}>"))
    all_tag_ids.add(proc.tokenizer.convert_tokens_to_ids(f"</s_{f}>"))

gseq = [dsi, start_id]
gparts = []
for f in HEADER_FIELDS:
    open_id  = proc.tokenizer.convert_tokens_to_ids(f"<s_{f}>")
    close_id = proc.tokenizer.convert_tokens_to_ids(f"</s_{f}>")
    seq_field = gseq + [open_id]
    banned = [[t] for t in all_tag_ids if t != close_id]
    with torch.no_grad():
        go = mdl.generate(
            pv, decoder_input_ids=torch.tensor([seq_field], device=dev),
            max_new_tokens=48, pad_token_id=proc.tokenizer.pad_token_id,
            eos_token_id=close_id, bad_words_ids=banned,
            no_repeat_ngram_size=3, num_beams=1, return_dict_in_generate=True,
        )
    gen = go.sequences[0].tolist()[len(seq_field):]
    value_ids = [t for t in gen if t not in all_tag_ids and t != proc.tokenizer.pad_token_id]
    value = proc.tokenizer.decode(value_ids, skip_special_tokens=True).strip()
    gparts.append(f"<s_{f}>{value}</s_{f}>")
    gseq = seq_field + value_ids + [close_id]

raw2 = f"{TASK_START_TOKEN}{''.join(gparts)}{TASK_END_TOKEN}"
print("RAW2 (reconstructed):\n", raw2[:800])
print("\nParsed2 ->", decode_donut_output(raw2, HEADER_FIELDS))



In [ ]:
# 5.2 Locate the run just produced, then score it
from pathlib import Path

runs = sorted(Path("results/runs").glob("*_donut_finetuned"), key=lambda p: p.stat().st_mtime)
LATEST_RUN_ID = runs[-1].name
print("Scoring run:", LATEST_RUN_ID)

!python -m src.eval.scorer --run-id {LATEST_RUN_ID}


In [ ]:
# 5.3 DEBUG — WHY is F1 0 though predictions are non-empty?
# Print PRED vs GOLD values field-by-field for a few docs. This shows whether the
# problem is (a) value formatting (leading spaces / truncation / OCR-ish drift) or
# (b) the scorer reading gold from a different place / key than our predictions.
import json
from pathlib import Path

results_file = Path("results/runs") / LATEST_RUN_ID / "zero_shot_results.jsonl"
with open(results_file) as f:
    preds = [json.loads(line) for line in f]

def load_gold(label_path):
    try:
        with open(label_path) as f:
            g = json.load(f)
        # labels may be nested (e.g. {"header": {...}}) or flat — try common shapes
        if isinstance(g, dict):
            for key in ("header", "fields", "extraction"):
                if key in g and isinstance(g[key], dict):
                    return g[key]
        return g
    except Exception as e:
        return {"<error>": str(e)}

for p in preds[:3]:
    print("=" * 70)
    print("doc:", p["doc_id"], "| level:", p["level"])
    pred = json.loads(p["prediction_raw"]) if p["prediction_raw"] else {}
    gold = load_gold(p["label_path"])
    print(f'label_path: {p["label_path"]}')
    print(f'{"FIELD":<20}{"PRED":<28}{"GOLD":<28}')
    keys = sorted(set(list(pred.keys()) + [k for k in gold.keys() if k != "<error>"]))
    for k in keys:
        pv = repr(pred.get(k))
        gv = repr(gold.get(k))
        mark = "OK" if pred.get(k) == gold.get(k) else "X"
        print(f"{k:<20}{pv:<28}{gv:<28}{mark}")

# Also show one raw gold file verbatim so we can see its exact structure/keys.
print("\n" + "=" * 70)
print("RAW GOLD FILE (verbatim) for first doc:")
print(Path(preds[0]["label_path"]).read_text()[:600])


In [ ]:
# 5.4 HONEST F1 — score ONLY the 10 trained HEADER_FIELDS.
# The repo scorer counts ALL ~20 gold fields (account_holder_address, account_type,
# branch_*, currency, customer_id, interest_rate, transactions, ...), but we only
# trained these 10 tags. Scoring on untrained fields drags F1 to ~0 no matter what.
# This cell restricts evaluation to what the model was actually taught, so we get a
# truthful per-field accuracy ceiling BEFORE investing in a larger data run.
import json, re
from pathlib import Path

HEADER_FIELDS = [
    "account_number", "ifsc_code", "micr_code", "bank_name", "account_holder",
    "opening_balance", "closing_balance", "start_date", "end_date", "statement_date",
]

results_file = Path("results/runs") / LATEST_RUN_ID / "zero_shot_results.jsonl"
with open(results_file) as f:
    preds = [json.loads(line) for line in f]

def load_gold(label_path):
    with open(label_path) as f:
        g = json.load(f)
    if isinstance(g, dict):
        for key in ("header", "fields", "extraction"):
            if key in g and isinstance(g[key], dict):
                return g[key]
    return g

def norm(v):
    # Normalize for a fair exact-match: strip tags/spaces, lowercase, drop trailing .0 on numbers.
    if v is None:
        return ""
    s = str(v)
    s = re.sub(r"<[^>]+>", "", s)          # remove any leaked <s_*>/</s_*> tags
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    m = re.fullmatch(r"(\d+)\.0", s)        # 1212467.0 -> 1212467 (json floats vs str)
    if m:
        s = m.group(1)
    return s

# Per-field tallies across all docs (micro-averaged).
tp = fp = fn = 0
per_field = {f: {"tp": 0, "fp": 0, "fn": 0} for f in HEADER_FIELDS}

for p in preds:
    pred = json.loads(p["prediction_raw"]) if p.get("prediction_raw") else {}
    gold = load_gold(p["label_path"])
    for f in HEADER_FIELDS:
        pv, gv = norm(pred.get(f)), norm(gold.get(f))
        if not gv:
            continue                         # gold empty -> field not applicable
        if pv == gv:
            tp += 1; per_field[f]["tp"] += 1
        else:
            fn += 1; per_field[f]["fn"] += 1
            if pv:
                fp += 1; per_field[f]["fp"] += 1

def prf(tp, fp, fn):
    p_ = tp / (tp + fp) if (tp + fp) else 0.0
    r_ = tp / (tp + fn) if (tp + fn) else 0.0
    f_ = 2 * p_ * r_ / (p_ + r_) if (p_ + r_) else 0.0
    return p_, r_, f_

print(f"HONEST header-only score on run: {LATEST_RUN_ID}  (docs={len(preds)})\n")
print(f'{"FIELD":<20}{"TP":>4}{"FP":>4}{"FN":>4}   F1')
for f in HEADER_FIELDS:
    d = per_field[f]
    _, _, ff = prf(d["tp"], d["fp"], d["fn"])
    print(f'{f:<20}{d["tp"]:>4}{d["fp"]:>4}{d["fn"]:>4}   {ff:.3f}')

P, R, F1 = prf(tp, fp, fn)
print("\n" + "=" * 40)
print(f"MICRO  precision={P:.3f}  recall={R:.3f}  F1={F1:.3f}")
print("=" * 40)
print("This is the true ceiling on the 10 trained fields. Use it as the baseline to")
print("beat after the bigger/more-diverse data run (more templates + higher --n).")


## 6. Validate with Assertions

Programmatic checks that fail loudly if the fix regressed:
- at least some predictions are non-empty and contain field tags
- not every document is `mostly_missing`

In [ ]:
# 6.1 Assertions on the fixed run's predictions
import json
from pathlib import Path

results_file = Path("results/runs") / LATEST_RUN_ID / "zero_shot_results.jsonl"
with open(results_file) as f:
    preds = [json.loads(line) for line in f]

n_total = len(preds)
n_error = sum(1 for p in preds if p["error"] is not None)

def _nonempty(p):
    if not p["prediction_raw"]:
        return False
    obj = json.loads(p["prediction_raw"])
    return any(v not in (None, "", []) for v in obj.values())

n_nonempty = sum(1 for p in preds if _nonempty(p))
print(f"total={n_total}  errors={n_error}  non_empty_predictions={n_nonempty}")

# Baseline was 0/20 non-empty. Fix must produce at least some populated predictions.
assert n_error == 0, f"{n_error} inference errors — check the loader fix"
assert n_nonempty > 0, "All predictions empty — fix did not resolve the 0.000-F1 bug"
print("PASS: fixed inference produces populated predictions (regression fixed).")


## 7. Quality Gate + Gated Scoring (turn the negative result into a shippable design)

**Finding:** accuracy tracks **image legibility**, not model choice — both this fine-tune
and an independent vision-LLM collapse on medium/heavy degraded scans. So the correct
product design is a **quality-gated pipeline**:

1. **Score each image** for legibility (Laplacian variance = focus/edge sharpness; higher =
   crisper text). Fast, no OCR needed, runs on CPU.
2. **Route:** `auto` (clean/light — readable) vs `review` (medium/heavy — defer to image
   restoration or a human). The system **abstains instead of hallucinating**.
3. **Gated F1:** re-run the honest header-only score **only on the `auto` bucket** and
   report **coverage** ("auto-process X% of docs at Y F1, defer the rest").

This reuses `data/splits/test.csv`, the latest run's predictions, and the Cell 5.4 scorer.
Run Section 5 first so `LATEST_RUN_ID` exists.


In [ ]:
# 7.1 Compute a legibility score per test image — DENOISE-then-score + text-aware proxy.
# WHY THIS CHANGED: raw Laplacian variance is fooled by noise. In the first run it ranked
# heavy (1001) ABOVE light (768) and medium (687) because JPEG/salt-pepper artifacts add
# fake high-frequency "edges". A quality gate built on a noise-fooled metric can't work.
# So here we (a) DENOISE first (bilateral: smooths noise, keeps text strokes), then take
# Laplacian variance; and (b) also compute an OCR mean-confidence proxy (Tesseract) which
# directly reflects readability. We evaluate BOTH and keep whichever ranks the tiers
# monotonically (clean > light > medium > heavy).
import csv, json
from pathlib import Path
import numpy as np
import cv2
from collections import defaultdict

results_file = Path("results/runs") / LATEST_RUN_ID / "zero_shot_results.jsonl"
with open(results_file) as f:
    preds = [json.loads(line) for line in f]

def _load_gray(image_path, target_h=1000):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    h, w = img.shape[:2]
    if h == 0:
        return None
    scale = target_h / h
    return cv2.resize(img, (max(1, int(w * scale)), target_h), interpolation=cv2.INTER_AREA)

def lap_raw(image_path):
    """OLD metric: variance of Laplacian on raw grayscale (noise-fooled). Kept for A/B."""
    g = _load_gray(image_path)
    return 0.0 if g is None else float(cv2.Laplacian(g, cv2.CV_64F).var())

def lap_denoised(image_path):
    """NEW metric: bilateral-denoise, THEN Laplacian variance. Bilateral removes speckle
    noise while preserving text edges, so heavy-degradation noise no longer inflates it."""
    g = _load_gray(image_path)
    if g is None:
        return 0.0
    d = cv2.bilateralFilter(g, d=7, sigmaColor=50, sigmaSpace=50)
    return float(cv2.Laplacian(d, cv2.CV_64F).var())

# OCR mean-confidence proxy (optional — needs pytesseract + tesseract binary).
try:
    import pytesseract
    _HAS_TESS = True
except Exception:
    _HAS_TESS = False

def ocr_conf(image_path):
    """Mean word-level OCR confidence — a direct readability signal. 0.0 if unavailable."""
    if not _HAS_TESS:
        return None
    g = _load_gray(image_path)
    if g is None:
        return 0.0
    try:
        data = pytesseract.image_to_data(g, output_type=pytesseract.Output.DICT)
        confs = [int(c) for c in data["conf"] if str(c).lstrip("-").isdigit() and int(c) >= 0]
        return float(np.mean(confs)) if confs else 0.0
    except Exception:
        return 0.0

# Compute all candidate metrics per doc.
for p in preds:
    p["q_lap_raw"]  = lap_raw(p["image_path"])
    p["q_lap_den"]  = lap_denoised(p["image_path"])
    p["q_ocr"]      = ocr_conf(p["image_path"])

def _tier_means(key):
    by = defaultdict(list)
    for p in preds:
        if p.get(key) is not None:
            by[p["level"]].append(p[key])
    return {lvl: float(np.mean(v)) for lvl, v in by.items()}, by

def _is_monotonic(means):
    order = ["clean", "light", "medium", "heavy"]
    vals = [means.get(t) for t in order if t in means]
    return all(a >= b for a, b in zip(vals, vals[1:])) and len(vals) == 4

def _report(name, key):
    means, by = _tier_means(key)
    if not by:
        print(f"[{name}] unavailable.\n")
        return means, False
    print(f"[{name}] by tier:")
    print(f'{"TIER":<10}{"n":>5}{"mean":>12}{"median":>12}{"min":>10}{"max":>10}')
    for lvl in sorted(by):
        v = np.array(by[lvl])
        print(f"{lvl:<10}{len(v):>5}{v.mean():>12.1f}{np.median(v):>12.1f}{v.min():>10.1f}{v.max():>10.1f}")
    mono = _is_monotonic(means)
    print(f"monotonic clean>light>medium>heavy? {'YES ✅' if mono else 'NO ❌'}  "
          f"(means: {', '.join(f'{k}={means[k]:.0f}' for k in ['clean','light','medium','heavy'] if k in means)})\n")
    return means, mono

print(f"Legibility metric A/B — run {LATEST_RUN_ID} (docs={len(preds)})\n")
_, mono_raw = _report("RAW Laplacian (old)", "q_lap_raw")
_, mono_den = _report("DENOISED Laplacian (new)", "q_lap_den")
if _HAS_TESS:
    _, mono_ocr = _report("OCR mean-confidence", "q_ocr")
else:
    mono_ocr = False
    print("[OCR mean-confidence] pytesseract not installed -> skipped. "
          "To enable: !apt-get install -y tesseract-ocr && pip install pytesseract\n")

# --- SELECTION CRITERION: rank-correlation with accuracy, NOT tier-monotonicity ---
# WHY: strict tier-monotonicity (clean>light>medium>heavy) is the WRONG success test.
# A gate only needs to push the ILLEGIBLE docs to low scores so raising the threshold
# RAISES auto-bucket F1. The DENOISED Laplacian does exactly that (clean=808 vs all
# degraded ~19-54), even though medium/heavy — both floored near zero — swap order.
# So we score each candidate metric by whether auto_F1 RISES as we tighten the gate.
def _quality_lift(key):
    """Return (auto_F1 at 90th pct) - (auto_F1 at 0), i.e. how much gating on this
    metric improves accuracy. Higher lift = better gate. Uses the same header-only
    exact-match scorer as cell 5.4/7.2 (defined inline to keep this cell standalone)."""
    import re as _re
    def _norm(v):
        if v is None: return ""
        s = _re.sub(r"<[^>]+>", "", str(v)).strip().lower()
        s = _re.sub(r"\s+", " ", s)
        m = _re.fullmatch(r"(\d+)\.0", s)
        return m.group(1) if m else s
    def _gold(lp):
        with open(lp) as f: g = json.load(f)
        if isinstance(g, dict):
            for kk in ("header", "fields", "extraction"):
                if kk in g and isinstance(g[kk], dict): return g[kk]
        return g
    def _f1(records):
        tp = fp = fn = 0
        for p in records:
            pr = json.loads(p["prediction_raw"]) if p.get("prediction_raw") else {}
            gd = _gold(p["label_path"])
            for f in ["account_number","ifsc_code","micr_code","bank_name","account_holder",
                      "opening_balance","closing_balance","start_date","end_date","statement_date"]:
                pv, gv = _norm(pr.get(f)), _norm(gd.get(f))
                if not gv: continue
                if pv == gv: tp += 1
                else:
                    fn += 1
                    if pv: fp += 1
        p_ = tp/(tp+fp) if (tp+fp) else 0.0
        r_ = tp/(tp+fn) if (tp+fn) else 0.0
        return 2*p_*r_/(p_+r_) if (p_+r_) else 0.0
    vals = np.array([p.get(key) if p.get(key) is not None else 0.0 for p in preds])
    base = _f1(preds)
    th90 = float(np.percentile(vals, 90))
    top = [p for i, p in enumerate(preds) if vals[i] >= th90]
    return (_f1(top) - base) if top else 0.0

candidates = [("q_lap_den", "DENOISED Laplacian"),
              ("q_lap_raw", "RAW Laplacian")]
if _HAS_TESS:
    candidates.append(("q_ocr", "OCR mean-confidence"))

print("Gate quality by ACCURACY LIFT (auto_F1@p90 - blended_F1); higher = better gate:")
lifts = {}
for key, name in candidates:
    lifts[key] = _quality_lift(key)
    print(f"  {name:<24} lift = {lifts[key]:+.4f}")
print()

# Choose the metric with the greatest positive lift; fall back to denoised Laplacian.
best_key = max(lifts, key=lifts.get)
if lifts[best_key] <= 0:
    best_key = "q_lap_den"
chosen_key = best_key
chosen_name = dict(candidates)[chosen_key]
gate_works = lifts.get(chosen_key, 0.0) > 0

for p in preds:
    p["quality"] = p[chosen_key] if p.get(chosen_key) is not None else 0.0

print("=" * 60)
print(f"CHOSEN legibility metric for gating: {chosen_name}  "
      f"(gate improves accuracy? {'YES ✅' if gate_works else 'NO ❌'})")
print("=" * 60)
all_q = np.array([p["quality"] for p in preds])
print(f"Overall: mean={all_q.mean():.1f}  median={np.median(all_q):.1f}  "
      f"p25={np.percentile(all_q,25):.1f}  p50={np.percentile(all_q,50):.1f}  "
      f"p75={np.percentile(all_q,75):.1f}")
print("The gate is judged by whether auto_F1 RISES as the threshold tightens (cells 7.2/7.3),"
      " NOT by strict tier-monotonicity — medium/heavy both floor near zero and may swap.")


In [ ]:
# 7.2 GATED SCORING — honest header-only F1 restricted to the readable ("auto") bucket.
# Reuses the exact norm()/PRF logic from cell 5.4, but only scores docs whose quality
# score >= a threshold. Sweeping the threshold traces the coverage/accuracy trade-off:
# defer the unreadable tail to review, and the auto bucket's F1 climbs well above the
# blended number. This is the shippable operating point.
import json, re
from pathlib import Path
import numpy as np

HEADER_FIELDS = [
    "account_number", "ifsc_code", "micr_code", "bank_name", "account_holder",
    "opening_balance", "closing_balance", "start_date", "end_date", "statement_date",
]

def load_gold(label_path):
    with open(label_path) as f:
        g = json.load(f)
    if isinstance(g, dict):
        for key in ("header", "fields", "extraction"):
            if key in g and isinstance(g[key], dict):
                return g[key]
    return g

def norm(v):
    if v is None:
        return ""
    s = re.sub(r"<[^>]+>", "", str(v)).strip().lower()
    s = re.sub(r"\s+", " ", s)
    m = re.fullmatch(r"(\d+)\.0", s)
    if m:
        s = m.group(1)
    return s

def prf(tp, fp, fn):
    p_ = tp / (tp + fp) if (tp + fp) else 0.0
    r_ = tp / (tp + fn) if (tp + fn) else 0.0
    f_ = 2 * p_ * r_ / (p_ + r_) if (p_ + r_) else 0.0
    return p_, r_, f_

def score_bucket(records):
    """Micro P/R/F1 over the 10 header fields for a list of prediction records."""
    tp = fp = fn = 0
    for p in records:
        pred = json.loads(p["prediction_raw"]) if p.get("prediction_raw") else {}
        gold = load_gold(p["label_path"])
        for f in HEADER_FIELDS:
            pv, gv = norm(pred.get(f)), norm(gold.get(f))
            if not gv:
                continue
            if pv == gv:
                tp += 1
            else:
                fn += 1
                if pv:
                    fp += 1
    return prf(tp, fp, fn)

# Sweep thresholds across the observed quality percentiles to show the trade-off.
qs = np.array([p["quality"] for p in preds])
thresholds = [0.0] + [float(np.percentile(qs, q)) for q in (25, 50, 60, 70, 75, 80, 90)]

print(f"GATED SCORING — run {LATEST_RUN_ID} (total docs={len(preds)})\n")
print(f'{"thresh":>10}{"coverage":>12}{"auto_docs":>11}{"auto_F1":>10}{"review_F1":>11}')
print("-" * 54)
for th in thresholds:
    auto   = [p for p in preds if p["quality"] >= th]
    review = [p for p in preds if p["quality"] <  th]
    if not auto:
        continue
    _, _, f_auto   = score_bucket(auto)
    _, _, f_review = score_bucket(review) if review else (0, 0, 0.0)
    cov = len(auto) / len(preds)
    print(f"{th:>10.1f}{cov:>11.1%}{len(auto):>11}{f_auto:>10.3f}{f_review:>11.3f}")

print("\nRead this as: 'auto-process COVERAGE of docs at AUTO_F1, defer the rest to review.'")
print("The auto_F1 should rise as the threshold rises (unreadable docs filtered out),")
print("while review_F1 stays near zero — proving deferral removes exactly the hopeless cases.")


In [ ]:
# 7.3 PICK AN OPERATING POINT + save a gated report (the presentable artifact).
# Strategy: choose the LOWEST threshold whose auto-bucket F1 is at least TARGET_F1 while
# still covering a useful share of docs. Falls back to the best-F1 threshold if none hit
# the target. Writes results/runs/<run>/gated_report.json for the write-up.
import json
from pathlib import Path
import numpy as np

TARGET_F1 = 0.05      # minimum acceptable auto-bucket F1 (tune to your SLA)
MIN_COVERAGE = 0.20   # don't recommend a point that auto-processes < 20% of docs

candidates = []
qs = np.array([p["quality"] for p in preds])
for th in [0.0] + [float(np.percentile(qs, q)) for q in range(10, 96, 5)]:
    auto = [p for p in preds if p["quality"] >= th]
    if not auto:
        continue
    _, _, f_auto = score_bucket(auto)
    cov = len(auto) / len(preds)
    candidates.append({"threshold": th, "coverage": cov, "auto_f1": f_auto, "auto_docs": len(auto)})

# Prefer points meeting BOTH target F1 and min coverage; pick the highest coverage among them.
qualified = [c for c in candidates if c["auto_f1"] >= TARGET_F1 and c["coverage"] >= MIN_COVERAGE]
if qualified:
    best = max(qualified, key=lambda c: c["coverage"])   # most docs auto-processed at acceptable F1
    reason = f"meets TARGET_F1>={TARGET_F1} and COVERAGE>={MIN_COVERAGE}"
else:
    best = max(candidates, key=lambda c: c["auto_f1"])   # otherwise just the most accurate gate
    reason = "no point met both targets -> highest-F1 gate (raise data quality to do better)"

th = best["threshold"]
auto   = [p for p in preds if p["quality"] >= th]
review = [p for p in preds if p["quality"] <  th]
Pa, Ra, Fa = score_bucket(auto)
Pr, Rr, Fr = score_bucket(review) if review else (0.0, 0.0, 0.0)
_, _, F_all = score_bucket(preds)

# Per-tier routing breakdown (how many of each degradation tier get auto vs deferred).
from collections import defaultdict
route = defaultdict(lambda: {"auto": 0, "review": 0})
for p in preds:
    route[p["level"]]["auto" if p["quality"] >= th else "review"] += 1

print("=" * 60)
print(f"RECOMMENDED OPERATING POINT ({reason})")
print("=" * 60)
print(f"quality threshold : {th:.1f}")
print(f"AUTO   : {len(auto):>4} docs ({len(auto)/len(preds):.1%})  F1={Fa:.3f}  P={Pa:.3f}  R={Ra:.3f}")
print(f"REVIEW : {len(review):>4} docs ({len(review)/len(preds):.1%})  (deferred to restoration/human)")
print(f"blended F1 (no gate, for reference): {F_all:.3f}")
print("\nRouting by degradation tier:")
print(f'{"TIER":<10}{"auto":>6}{"review":>8}')
for lvl in sorted(route):
    print(f'{lvl:<10}{route[lvl]["auto"]:>6}{route[lvl]["review"]:>8}')

report = {
    "run_id": LATEST_RUN_ID,
    "total_docs": len(preds),
    "target_f1": TARGET_F1,
    "min_coverage": MIN_COVERAGE,
    "recommended_threshold": th,
    "selection_reason": reason,
    "auto": {"docs": len(auto), "coverage": len(auto)/len(preds),
             "precision": Pa, "recall": Ra, "f1": Fa},
    "review": {"docs": len(review), "coverage": len(review)/len(preds) if preds else 0.0,
               "f1": Fr},
    "blended_f1_no_gate": F_all,
    "routing_by_tier": {lvl: dict(v) for lvl, v in route.items()},
    "sweep": candidates,
}
out_path = Path("results/runs") / LATEST_RUN_ID / "gated_report.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"\nSaved gated report -> {out_path}")
print("Pitch: 'We auto-extract the readable subset at the reported F1 and safely defer the")
print("degraded tail to image-restoration/human review, instead of emitting hallucinations.'")
